# Milepost — serving the model to a program

**Milepost** is a used-car marketplace. Two models are finished and sitting on disk: one
estimates what a car is worth today, the other pre-checks whether a buyer will clear finance.

A browser app in front of them is a solved problem. This is the other request, and it is the
one that arrives by email: *"We are the partner bank. Send us an endpoint and we will call it
from our loan desk."*

That sentence rules out every UI framework in existence, because a UI answers a human and a
bank answers with code. What they are asking for is a **URL that takes JSON and returns JSON** —
and the whole difficulty is not producing one. It is producing one that behaves when the JSON
is wrong.

| | |
|---|---|
| **Flask** | P3 – P4. A route is a decorator on a function. That is very nearly the whole framework — and when a payload is malformed, it is very nearly the whole problem. |
| **FastAPI** | P5 – P7. The same two endpoints, with the request declared as a Python type. The framework then refuses bad requests before your function runs, and documents the contract for the caller. |

## The map

```
            MILEPOST — one model layer, two services, one contract

  data/                          P1                      artifacts/
  ─────                          ──                      ──────────
  cars24-car-price.csv ──► XGBRegressor        ──────► price_model.joblib
  train_flask.csv      ──► LogisticRegression  ──────► loan_model.joblib
                                 │                      model_meta.json
                                 └── the decision threshold ───┘  (a SERVING choice)
                                                             │
        P2  how the web works — DNS · HTTP · status codes · GET vs POST
                                                             │
      ┌──────────────────────────────────────────────────────┘
      │   service      file              a bad request is     the caller's docs
      │   ───────      ────              ────────────────     ─────────────────
      ├─ P3/P4 FLASK   flask_app.py      500 — YOUR fault     whatever you write
      └─ P5    FASTAPI fastapi_app.py    422 — THEIR fault    generated, live at /docs

      P6  the seam    P7  async, honestly    P8  workers    P9  pick one, then ship it
```

Both service files are sitting in this folder already. The notebook never writes them — it
points at them, shows the handful of lines that carry each lesson, and calls them.

And it never starts a server. Flask and FastAPI both hand you an in-process client that runs
the real routing stack with no port bound, which is what every cell below uses. The
`flask run` / `uvicorn` / `curl` commands are fenced blocks, for a terminal you own and can
close.

In [1]:
%pip install -q flask fastapi "uvicorn[standard]" gunicorn requests httpx scikit-learn xgboost joblib pandas numpy ipython-autotime


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autotime

time: 112 µs (started: 2026-09-23 20:09:06 +05:30)


In [3]:
import json
import os
import shutil
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

DATA = Path("data")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("future.no_silent_downcasting", True)
warnings.filterwarnings("ignore", category=FutureWarning)

import fastapi as _fastapi, flask as _flask, pydantic as _pydantic
import sklearn as _sklearn, xgboost as _xgboost

# Defaults, APIs and error shapes all move between releases. Pin what this ran on.
print("versions        :", f"flask {_flask.__version__} · fastapi {_fastapi.__version__} "
      f"· pydantic {_pydantic.__version__} · scikit-learn {_sklearn.__version__} "
      f"· xgboost {_xgboost.__version__}")
print("data files      :", sorted(p.name for p in DATA.glob("*.csv")))
print("service files   :", sorted(p.name for p in Path(".").glob("*_app.py")))
print("artifacts so far:", sorted(p.name for p in ARTIFACTS.glob("*")) or "(empty — Part 1 fills this)")

versions        : flask 3.1.3 · fastapi 0.139.0 · pydantic 2.12.5 · scikit-learn 1.7.2 · xgboost 3.2.0
data files      : ['cars24-car-price.csv', 'train_flask.csv']
service files   : ['fastapi_app.py', 'flask_app.py']
artifacts so far: ['loan_model.joblib', 'model_meta.json', 'price_model.joblib']
time: 1.33 s (started: 2026-09-23 20:09:06 +05:30)


/var/folders/p6/6_nprx9x22s4njm57gzb34l40000gp/T/ipykernel_55700/700244308.py:26: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("versions        :", f"flask {_flask.__version__} · fastapi {_fastapi.__version__} "


```
P0 setup  [ P1 MODELS ]  P2 the web  P3 flask  P4 flask's gaps  P5 fastapi  P6 the seam  P7 async  P8 workers  P9 pick one
```

# Part 1 — The models, and who is asking for them

Nothing here is new modelling work. The goal is the three files every later Part loads, and
two ideas that cause more production incidents than any algorithm choice:

**There are two models because a service is not one shape.** The price estimator is a
**regressor**: its answer is a number and its fields are numeric and ranged, so it is the one that
can be sent a wrong *type* (§4.2), priced in bulk (§5.5), timed (§7.1), and used as a startup
canary (§8.1) — a number moves when the artifact underneath it changes. The finance pre-check is a
**classifier**: its answer is a *decision* and its fields are fixed categories, so it is the one
that carries the judgement calls — a correct value silently bucketed into an `else` (§4.3, §4.4)
and who owns the cut that turns a probability into a verdict (§5.6). Neither stands in for the
other: there is no threshold on a price, and `"Loan Approved"` is a canary that would survive
being served by the wrong model.

> **Whatever you did to the columns at training time, you must do again at request time —
> exactly, in the same order.**
>
> **And every decision the serving code makes that the model cannot make for itself has to be
> written down somewhere it will travel.**

## 1.1 Why a browser app cannot take this call

Put a widget-based UI in front of the price model and it works, for a person. Now read the
partner bank's request again and check it against what that UI can do:

- There is **no URL that returns a price.** A UI server answers `http://localhost:8501` with a JavaScript application — HTML, a bundle, a websocket handshake — not `{"price_lakhs": 4.87}`.
- There is **no contract.** Nothing anywhere declares which fields exist, which are required, or what types they take. The knowledge lives in the widget definitions and in your head.
- There is **no way to call it from code.** Those frameworks assume a browser driving a websocket, not a `requests.post`.
- It is **stateful per user.** Each browser tab gets its own Python session holding its own copy of the model. That is fine for ten analysts and is not a model for a service under load.

These are not missing features that a later release will add. They are consequences of being
aimed at a human. Serving a program is a different tool.

## 1.2 The environment is part of the deliverable

The bank is not going to receive your laptop. Whatever you hand over has to rebuild elsewhere,
and a `.joblib` file on its own does not — it is a **pickled Python object graph**, which has
three consequences worth internalising before shipping one:

1. **It carries no code.** Unpickling reconstructs `XGBRegressor` by importing `xgboost`. No xgboost on the server, no model.
2. **It is version-sensitive.** A model pickled under scikit-learn 1.7 and loaded under 1.3 may warn, may raise, may silently behave differently.
3. **Unpickling executes code.** Never `joblib.load()` a file you did not produce.

So the artifact travels with a pinned environment or it does not travel at all:

```bash
# create and activate — venv ships with Python
python -m venv .venv && source .venv/bin/activate

# freeze what you actually have, so the other machine can rebuild it
pip freeze > requirements.txt
```

`requirements.txt` in this folder is pinned with `==`, not `>=`, for exactly reason 2. The
formats that avoid all three — ONNX, or a framework's own `save_model` — cost more effort up
front; for a service where you control both ends, a pickle plus pinned versions is fine.

## 1.3 The price estimator

19,980 used-car listings. `selling_price` is in lakhs of rupees.

In [4]:
cars = pd.read_csv(DATA / "cars24-car-price.csv")
print(cars.shape)
cars.head()

(19980, 11)


,full_name,selling_price,year,seller_type,km_driven,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Maruti Alto Std,1.20,2012.0,Individual,120000,Petrol,Manual,19.70,796.0,46.30,5.0
1,Hyundai Grand i10 Asta,5.50,2016.0,Individual,20000,Petrol,Manual,18.90,1197.0,82.00,5.0
2,Hyundai i20 Asta,2.15,2010.0,Individual,60000,Petrol,Manual,17.00,1197.0,80.00,5.0
3,Maruti Alto K10 2010-2014 VXI,2.26,2012.0,Individual,37000,Petrol,Manual,20.92,998.0,67.10,5.0
4,Ford Ecosport 2015-2021 1.5 TDCi Titanium BSIV,5.70,2015.0,Dealer,30000,Diesel,Manual,22.77,1498.0,98.59,5.0


time: 23.5 ms (started: 2026-09-23 20:09:07 +05:30)


Three columns are text, and the regressor only accepts numbers, so each one gets a fixed
integer code.

The dictionary below is the **contract between training and serving**. Both services later in
this notebook carry an identical copy — if one of them ever encodes `Petrol` as `1` while
training encoded it as `2`, nothing raises, nothing logs, and every petrol car is priced as a
diesel.

In [5]:
encode_dict = {
    "fuel_type": {"Diesel": 1, "Petrol": 2, "CNG": 3, "LPG": 4, "Electric": 5},
    "transmission_type": {"Manual": 1, "Automatic": 2},
    "seller_type": {"Dealer": 1, "Individual": 2, "Trustmark Dealer": 3},
}

PRICE_FEATURES = [
    "year",
    "seller_type",
    "km_driven",
    "fuel_type",
    "transmission_type",
    "mileage",
    "engine",
    "max_power",
    "seats",
]

df = cars.drop(columns=["full_name"]).replace(encode_dict).infer_objects(copy=False)
X, y = df[PRICE_FEATURES], df["selling_price"]
X.head()

,year,seller_type,km_driven,fuel_type,transmission_type,mileage,engine,max_power,seats
0,2012.0,2,120000,2,1,19.70,796.0,46.30,5.0
1,2016.0,2,20000,2,1,18.90,1197.0,82.00,5.0
2,2010.0,2,60000,2,1,17.00,1197.0,80.00,5.0
3,2012.0,2,37000,2,1,20.92,998.0,67.10,5.0
4,2015.0,1,30000,1,1,22.77,1498.0,98.59,5.0


time: 13.4 ms (started: 2026-09-23 20:09:07 +05:30)


The six settings below came out of a hyperparameter search. Reproducing that search is not
what this notebook is about — serving its result is — so they are written down as literals,
which is also exactly how they would reach a deployment.

No scaler. A tree splits on thresholds, so rescaling a column cannot move where those
thresholds fall; the model is the whole artifact.

In [6]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

BEST_PARAMS = {
    "n_estimators": 404,
    "max_depth": 7,
    "learning_rate": 0.06607192356835712,
    "subsample": 0.994398722227957,
    "colsample_bytree": 0.6118312727750044,
    "reg_lambda": 0.2758980370254515,
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

price_model = XGBRegressor(random_state=42, **BEST_PARAMS).fit(X_train, y_train)

print(f"test MAE : {mean_absolute_error(y_test, price_model.predict(X_test)):.4f} lakhs   <- goes into model_meta.json")

test MAE : 0.9815 lakhs   <- goes into model_meta.json
time: 1.12 s (started: 2026-09-23 20:09:07 +05:30)


## 1.4 The finance pre-check

614 historical applications with the outcome attached. Same discipline: map the text columns,
remember the mapping, save the model.

In [7]:
loans = pd.read_csv(DATA / "train_flask.csv")
print(loans.shape)
loans.head()

(614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


time: 5.2 ms (started: 2026-09-23 20:09:08 +05:30)


In [8]:
loan_encode_dict = {
    "Gender": {"Male": 0, "Female": 1},
    "Married": {"No": 0, "Yes": 1},
    "Credit_History": {"Uncleared Debts": 0, "Cleared Debts": 1},
}

LOAN_FEATURES = ["Gender", "Married", "ApplicantIncome", "LoanAmount", "Credit_History"]

loans["Gender"] = loans["Gender"].map(loan_encode_dict["Gender"])
loans["Married"] = loans["Married"].map(loan_encode_dict["Married"])
loans["Loan_Status"] = loans["Loan_Status"].map({"N": 0, "Y": 1})

loans = loans.dropna(subset=LOAN_FEATURES + ["Loan_Status"])

X_loan, y_loan = loans[LOAN_FEATURES], loans["Loan_Status"]
print("rows after dropna:", X_loan.shape[0], "of 614")
print("approval rate    :", round(float(y_loan.mean()), 3))

rows after dropna: 529 of 614
approval rate    : 0.69
time: 2.01 ms (started: 2026-09-23 20:09:08 +05:30)


Note the encoding for `Credit_History`: the CSV already holds `1.0` for a clean record and
`0.0` for debts, and the dictionary above says the same thing in words —
`"Cleared Debts" -> 1`, `"Uncleared Debts" -> 0`. Both services accept the words and convert.
Remember that those two strings are the only two spellings that mean anything; P4 is about what
happens to a third.

In [9]:
from sklearn.linear_model import LogisticRegression

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_loan, y_loan, test_size=0.2, random_state=42, stratify=y_loan
)

loan_model = LogisticRegression(max_iter=1000).fit(Xl_train, yl_train)

print("train accuracy:", round(loan_model.score(Xl_train, yl_train), 3))
print("test  accuracy:", round(loan_model.score(Xl_test, yl_test), 3))

train accuracy: 0.811
test  accuracy: 0.849
time: 66.8 ms (started: 2026-09-23 20:09:08 +05:30)


`max_iter=1000` rather than the default 100: `ApplicantIncome` runs into the tens of thousands
while `Credit_History` is 0 or 1, and lbfgs cannot converge across that range in 100 steps. The
default emits a `ConvergenceWarning` and hands back a half-fitted model that still happily
predicts. It does not crash; it is just wrong — the same failure shape as an encoding mismatch,
and the reason this notebook keeps measuring things instead of trusting them.

### One field decides this model

Worth knowing before P4, because it is what makes P4's failure expensive. `Credit_History` is
the input that carries the answer. Flip it and the verdict flips with it:

In [10]:
buyer_row = pd.DataFrame([[0, 1, 5000.0, 128.0, 1]], columns=LOAN_FEATURES)

for label, credit in [("Cleared Debts", 1), ("Uncleared Debts", 0)]:
    row = buyer_row.copy()
    row["Credit_History"] = credit
    verdict = "Loan Approved" if loan_model.predict(row)[0] == 1 else "Loan Rejected"
    print(f"{label:<18} -> {verdict}")

Cleared Debts      -> Loan Approved
Uncleared Debts    -> Loan Rejected
time: 1.12 ms (started: 2026-09-23 20:09:08 +05:30)


Same buyer, same income, same requested amount. One field, both outcomes. Hold on to that.

## 1.5 Three files on disk

That is the modelling done — except that one number still has nowhere to live.

`loan_model.predict(...)` was used twice above, and both times it called anything at probability
0.5 or higher an approval. **That 0.5 is not a default anybody chose.** There is no threshold
argument in the call; 0.5 is simply where `argmax` over two columns happens to flip.

Milepost's finance desk works to **0.80**. That number is not something the model can be asked
to remember — it moves when the business changes its mind about how many pre-checks a day it can
process, and no retraining is involved when it does. So it ships as data, beside the artifacts.

Part 5 comes back to who owns that number in production, and Part 4 prices what happens when a
service forgets to read it.

In [11]:
import sklearn

DECISION_THRESHOLD = 0.80
MODEL_VERSION = "milepost-2026-09-23"

joblib.dump(price_model, ARTIFACTS / "price_model.joblib")
joblib.dump(loan_model, ARTIFACTS / "loan_model.joblib")

(ARTIFACTS / "model_meta.json").write_text(json.dumps({
    "model_version": MODEL_VERSION,
    "decision_threshold": DECISION_THRESHOLD,
    "price_features": PRICE_FEATURES,
    "loan_features": LOAN_FEATURES,
    "price_test_mae_lakhs": round(float(mean_absolute_error(y_test, price_model.predict(X_test))), 4),
    "sklearn_version": sklearn.__version__,
}, indent=2))

for p in sorted(ARTIFACTS.glob("*")):
    print(f"  {p.name:<24} {p.stat().st_size / 1024:8.1f} KB")

  loan_model.joblib             1.3 KB
  model_meta.json               0.4 KB
  price_model.joblib         2470.8 KB
time: 7.78 ms (started: 2026-09-23 20:09:08 +05:30)


Note the sizes: 400 trees of depth 7, against five logistic coefficients, against a few hundred
bytes of JSON. The API in front of them cannot tell the difference — which is the first quietly
useful thing about putting an API in front of a model.

`model_meta.json` is small and it is not optional. It carries the threshold, the model version,
the feature order, and the scikit-learn version that pickled the other two files, so a service
can check on startup that it is about to load artifacts it understands. Both services read all
three files, and both refuse to start if any of them is missing.

There is one holdout number in there, for the price model, and **no accuracy for the loan
model** — which is deliberate. `price_test_mae_lakhs` is a record of how good that model is, and
nothing in this file can change it. An accuracy would not be: it moves when
`decision_threshold` moves, so a number stored *beside* the threshold cannot also be a record of
model quality. Written with `loan_model.score(...)` it would have said **0.8491**, which is the
accuracy of a service running at 0.5 — not the one this file configures. Measured honestly at
0.80 it says 0.6415, and that number is lower mostly because a cautious cut declines people who
would have paid, which is the outcome the finance desk asked for.

The rule worth taking from that: **store what the service checks, not what looks good.** A
service reads this file to confirm it understands its artifacts — versions, feature order, the
cut it must apply. A metric that depends on one of those values is a registry's job, not a
config file's.

```
P0 setup  P1 models  [ P2 THE WEB ]  P3 flask  P4 flask's gaps  P5 fastapi  P6 the seam  P7 async  P8 workers  P9 pick one
```

# Part 2 — How the web actually works

Skip this and Flask is magic incantations. Four ideas, then both frameworks make sense on sight.

## 2.1 A name is not an address

`www.google.com` is a label for humans. Machines route to IP addresses. **DNS** is the phone
book that converts one to the other, and the answer is cached for a TTL (time to live) — which
is why a freshly-pointed domain takes minutes to work everywhere.

| Type | Domain name | IP address | TTL |
|---|---|---|---|
| A | www.google.com | 172.253.122.99 | 5 min |
| A | www.google.com | 172.253.122.103 | 5 min |
| A | www.google.com | 172.253.122.104 | 5 min |

Several addresses for one name is not a mistake — that is load balancing, handed out by the
phone book itself.

In [12]:
import socket

for host in ["localhost", "www.google.com"]:
    try:
        print(f"{host:<18} -> {socket.gethostbyname(host)}")
    except OSError as exc:
        print(f"{host:<18} -> lookup failed ({exc})")

localhost          -> 127.0.0.1
www.google.com     -> 142.251.156.119
time: 34 ms (started: 2026-09-23 20:09:08 +05:30)


`localhost` resolves to `127.0.0.1` without touching the network at all — it is hard-coded on
every machine. That is the address both services in this notebook bind to, and the reason
`127.0.0.1:5001` on your laptop is not reachable by the bank.

## 2.2 HTTP: the rules everyone agreed to

A client sends a **request**, a server sends back a **response**. Both carry headers (metadata)
and usually a body (the payload).

```mermaid
sequenceDiagram
    participant C as Partner bank
    participant S as milepost.com
    C->>S: POST /predict/price<br/>Content-Type: application/json<br/>{"year": 2018, ...}
    S-->>C: 200 OK<br/>Content-Type: application/json<br/>{"price_lakhs": 4.87}
```

The header that matters most for a model API is **`Content-Type: application/json`** — it is how
the client tells the server "the bytes in this body are JSON, parse them as such". Send JSON
without it and Flask's `request.get_json()` hands your handler `None`, and the first line that
subscripts it raises.

### The status code, in one glance

The first digit is the whole story:

| Range | Means | You will meet |
|---|---|---|
| `1xx` | Hold on | (rare) |
| `2xx` | Here you go | **200 OK**, 201 Created |
| `3xx` | Go away (look over there) | 301/302 redirect, 307 |
| `4xx` | **You** screwed up | **404** wrong URL, **405** wrong method, **422** bad payload, 401/403 auth |
| `5xx` | **I** screwed up | **500** unhandled exception in your code |

The `4xx` / `5xx` split is not cosmetic — it decides whose pager goes off. A malformed request
answered with `500` tells the bank's monitoring that *Milepost* is broken, and your on-call gets
woken up for someone else's typo. The same request answered with `422` tells the caller to fix
their payload. **P4 produces the first. P5 produces the second.** That is the arc of this
notebook, and it is a two-character difference in the status line.

## 2.3 GET or POST

| | `GET` | `POST` |
|---|---|---|
| Purpose | Fetch something | Send something that causes work |
| Payload | In the URL (`?a=1&b=2`) | In the request **body** |
| Size limit | ~2 KB of URL | Effectively none |
| Safe to repeat | Yes — changes nothing | No — may act twice |
| Browsing a profile | ✅ | |
| Publishing a post | | ✅ |

A prediction endpoint is a `POST`. Not because it changes data — usually it does not — but
because the input is a structured object with nine fields, and structured objects belong in a
body, not smuggled through a URL.

In [13]:
from urllib.parse import parse_qs, urlsplit

url = "https://api.milepost.com/v1/predict/price?currency=inr&explain=true"
parts = urlsplit(url)

print(f"scheme   {parts.scheme}")
print(f"host     {parts.netloc}        <- the base URL: https://{parts.netloc}")
print(f"path     {parts.path}          <- the endpoint")
print(f"query    {parse_qs(parts.query)}")

scheme   https
host     api.milepost.com        <- the base URL: https://api.milepost.com
path     /v1/predict/price          <- the endpoint
query    {'currency': ['inr'], 'explain': ['true']}
time: 275 µs (started: 2026-09-23 20:09:08 +05:30)


**Base URL + endpoint.** One server (`api.milepost.com`) hosts many endpoints
(`/v1/predict/price`, `/v1/predict/loan`, `/health`). Building an API is deciding what those
endpoints are and what each one accepts — which is the next three Parts.

## 2.4 What an API is

A restaurant. You (the client) never enter the kitchen. You read a **menu** — the fixed list of
things you may ask for. You tell the **waiter** (the API), who knows how to carry your order.
The **kitchen** (the model) does the work. Food comes back.

The menu is the contract. It is why you cannot order something the kitchen has never heard of,
and why the kitchen can be completely rebuilt without you learning a new way to order.

```mermaid
flowchart LR
    U["Client<br/>(browser, bank, script)"] -- request --> A["API<br/>routes + contract"]
    A -- calls --> M["model_pred / loan_pred"]
    M -- value --> A
    A -- response --> U
```

**`curl`** is the plainest possible client — no framework, no browser, nothing hidden. If an
endpoint works in `curl`, it works.

```
P0 setup  P1 models  P2 the web  [ P3 FLASK ]  P4 flask's gaps  P5 fastapi  P6 the seam  P7 async  P8 workers  P9 pick one
```

# Part 3 — Flask: the surface for another program

Flask is small on purpose. A Flask app is an object, and you attach functions to URLs with a
decorator. That is very nearly the entire framework.

```python
app = Flask(__name__)                      # the application object

@app.route("/monday", methods=["GET"])     # when a GET arrives at /monday...
def monday_endpoint():                     # ...run this function
    return "No!!! It is Monday!"           # ...and send back what it returns
```

Three details that trip people up:

- **`__name__`** tells Flask where the app lives so it can find files next to it. It is `"__main__"` when you run the file directly, and the module name when it is imported.
- **The function name is irrelevant.** The *route* is the address. Two handlers cannot share a Python name, but nothing else about the name matters.
- **Return a `dict` and Flask serialises it to JSON** and sets `Content-Type: application/json` for you. Return a string and you get HTML.

## 3.0 Start it first, in a terminal you own

None of the cells below need a server — `test_client` in §3.2 answers requests in this process.
Postman does need one, so start it now, before reading the file:

```bash
flask --app flask_app.py run --port 5001
```

It prints `Running on http://127.0.0.1:5001`. Leave it running, and **keep that terminal where you
can see it** — when a request fails, the traceback appears there and not in the response. `Ctrl-C`
stops it.

**Why `--port` and not the default.** Flask picks **5000**, and on macOS that port belongs to
Control Centre's AirPlay Receiver, so the bind fails for a reason that has nothing to do with your
code. Naming the port is a habit worth having.

**In Postman**, every request in this Part is the same four steps:

| | |
|---|---|
| Method | `POST` — or `GET` for `/`, `/monday` and `/health` |
| URL | `http://127.0.0.1:5001/predict/loan` |
| Body | **raw**, then choose **JSON** in the dropdown beside it |
| Send | the JSON body is all that is needed: no auth, no query parameters, no other headers |

Choosing **JSON** in that dropdown is what sets `Content-Type: application/json`, and it is not
cosmetic. Leave it on **Text** and Flask will not attempt to parse the body at all — it answers
**415 Unsupported Media Type** before your handler runs, which is a different lesson than the one
you were aiming at.

## 3.1 The file

Open **`flask_app.py`**. It is one file, in this folder, and it has three layers:

1. **The model layer** — `encode_dict`, `loan_encode_dict`, `PRICE_FEATURES`, `LOAN_FEATURES`, the artifact loads, `THRESHOLD`, `model_pred`, `loan_pred`. Identical in intent to Part 1, written out again because a service is not allowed to depend on a notebook having run.
2. **The app object** — one line.
3. **The routes** — six decorators.

The loads sit in the **module body**, which Python runs exactly once per process:

```python
price_model = joblib.load(ARTIFACTS / "price_model.joblib")
loan_model = joblib.load(ARTIFACTS / "loan_model.joblib")
META = json.loads((ARTIFACTS / "model_meta.json").read_text())

THRESHOLD = META["decision_threshold"]
if not 0.0 < THRESHOLD < 1.0:
    raise ValueError(f"decision_threshold {THRESHOLD!r} is not a probability")
MODEL_VERSION = META["model_version"]
```

Not inside the handler. A `joblib.load()` per request would re-read 2.4 MB from disk and rebuild
400 trees on every call, and it would do it while the caller waits. Load at import, answer from
memory. P8 is about what that costs when you run four of these processes.

`THRESHOLD` is §1.5's number, read in the one place that needs it. `loan_pred` then calls
`predict_proba` and compares — not `predict`, which would silently mean 0.5 again.

The two lines after it are deliberate: a threshold outside `(0, 1)`, or a `model_meta.json` with
no threshold in it at all, must stop this process from starting. There is no default to fall back
to, because 0.5 is not a default — it is where `argmax` happens to flip. §5.6 argues for that
and P8 shows it happening.

And the routes themselves, with the bodies stripped out:

```python
@app.route("/", methods=["GET"])                      def index()           -> HTML string
@app.route("/monday", methods=["GET"])                def monday_endpoint() -> plain string
@app.route("/health", methods=["GET"])                def health()          -> dict, becomes JSON
@app.route("/predict/price", methods=["POST"])        def predict_price()
@app.route("/predict/price/batch", methods=["POST"])  def predict_price_batch()
@app.route("/predict/loan",  methods=["POST"])        def predict_loan()
```

In [14]:
import flask_app

# url_map is Flask's own routing table -- the decorators, read back.
print(flask_app.app.url_map)

Map([<Rule '/static/<filename>' (HEAD, OPTIONS, GET) -> static>,
 <Rule '/' (HEAD, OPTIONS, GET) -> index>,
 <Rule '/monday' (HEAD, OPTIONS, GET) -> monday_endpoint>,
 <Rule '/health' (HEAD, OPTIONS, GET) -> health>,
 <Rule '/predict/price' (POST, OPTIONS) -> predict_price>,
 <Rule '/predict/price/batch' (POST, OPTIONS) -> predict_price_batch>,
 <Rule '/predict/loan' (POST, OPTIONS) -> predict_loan>])
time: 5.29 ms (started: 2026-09-23 20:09:08 +05:30)


Seven rules, and only six of them were written by anyone. `/static/<filename>` is Flask's own,
registered because `Flask(__name__)` told it where the app lives and therefore where a `static/`
folder would be if one existed — the first of those three gotchas, showing up in the routing
table.

`HEAD` and `OPTIONS` next to every `GET` are Flask's too, added without being asked: `HEAD` is a
`GET` with the body left off, and `OPTIONS` answers *which methods does this address accept*. The
routes you wrote are the ones with `POST` on them and the three `GET`s.

Importing the module was also enough to load both models — the `FileNotFoundError` guard at the
top of the file is what turns a missing artifact into a refusal to start, rather than a service
that boots happily and answers every request wrongly.

## 3.2 Calling it without starting a server

The normal way to poke a Flask API by hand is Postman, or `curl`, or a browser — and that is
still the right tool for *exploring* a service that is already running. It is not the tool used
below, and the reason is worth being clear about.

`app.test_client()` is not a replacement for Postman. It is a replacement for **pytest**.

A Flask app object can serve requests directly, in-process, with no port bound and nothing to
shut down afterwards. `test_client` is the real routing stack — the same decorators, the same
JSON handling, the same status codes — just without the network.

| | Postman · curl · `/docs` | `test_client` |
|---|---|---|
| needs a process listening on a port | yes | **no** |
| one request at a time, driven by hand | yes | it is code, so a hundred if you want |
| can reach the Python objects inside the service | no | **yes** |
| runs unattended, in CI, on every commit | no | **yes** |

The third row is what the rest of this notebook needs. P6 scores the same input through the
endpoint *and* by calling the model directly in the same process, then diffs the two — Postman
cannot see `loan_model`. §5.6 forces the threshold to `0.5` mid-cell to show a verdict flip, and
P8 sets the loaded model to `None` to make `/health` answer `503`. All of that is inside the
process, and Postman is outside by definition.

So: Postman to explore, `test_client` to prove. The terminal-and-`curl` path is in §3.4, and it
is the one to use once the service is up.

In [15]:
client = flask_app.app.test_client()

r = client.get("/")
print("GET /           ->", r.status_code, r.headers["Content-Type"])
print(r.get_data(as_text=True)[:120].strip(), "...")

GET /           -> 200 text/html; charset=utf-8
<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>Milepost API</title>
<style>
  body { font-family: ...
time: 1.54 ms (started: 2026-09-23 20:09:08 +05:30)


In [16]:
r = client.get("/monday")
print("GET /monday           ->", r.status_code, "|", r.get_data(as_text=True))

r = client.get("/health")
print("GET /health           ->", r.status_code, "|", r.get_json())

r = client.get("/predict/price")
print("GET /predict/price    ->", r.status_code, "  <- 405: right URL, wrong method")

r = client.get("/no-such-endpoint")
print("GET /no-such-endpoint ->", r.status_code, "  <- 404: no route matches")

GET /monday           -> 200 | No!!! It is Monday!
GET /health           -> 200 | {'canary_price_lakhs': 5.31, 'decision_threshold': 0.8, 'model_version': 'milepost-2026-09-23', 'models': ['price', 'loan'], 'sklearn_version': '1.7.2', 'sklearn_version_at_train': '1.7.2', 'status': 'ok'}
GET /predict/price    -> 405   <- 405: right URL, wrong method
GET /no-such-endpoint -> 404   <- 404: no route matches
time: 3.02 ms (started: 2026-09-23 20:09:08 +05:30)


Four status codes from the table in P2, produced by a real app rather than asserted. Note
`/health` returning `application/json` without a single line of serialisation code — that is the
`dict` rule doing its work.

And note what it answered *with*. It did not say `{"status": "ok"}`; it scored a fixed canary
car, reported the threshold it is applying and named the scikit-learn version that pickled the
artifacts against the one it is running. P8 is about why the short version of that endpoint is
close to useless.

Now the two endpoints that matter.

In [17]:
car = {
    "year": 2018,
    "seller_type": "Dealer",
    "km_driven": 45000,
    "fuel_type": "Petrol",
    "transmission_type": "Manual",
    "mileage": 18.5,
    "engine": 1200,
    "max_power": 85.0,
    "seats": 5,
}

r = client.post("/predict/price", json=car)
print(r.status_code, r.headers["Content-Type"])
print(r.get_json())

200 application/json
{'model_version': 'milepost-2026-09-23', 'price_lakhs': 5.31}
time: 2 ms (started: 2026-09-23 20:09:08 +05:30)


In [18]:
buyer = {
    "Gender": "Male",
    "Married": "No",
    "Credit_History": "Cleared Debts",
    "ApplicantIncome": 5000,
    "LoanAmount": 128,
}

r = client.post("/predict/loan", json=buyer)
print(r.status_code, r.get_json())

200 {'loan_approval_status': 'Loan Approved', 'model_version': 'milepost-2026-09-23', 'probability': 0.8359, 'threshold_applied': 0.8}
time: 1.01 ms (started: 2026-09-23 20:09:08 +05:30)


Two models, one base URL, two endpoints, no UI anywhere. `json=` on the client is what sets
`Content-Type: application/json`; drop it and `request.get_json()` returns `None` and the
handler raises on the first subscript.

## 3.3 The same request, from a real client

`test_client` proves the routing. What the bank will actually write is `requests`, and the shape
is worth seeing side by side — the dictionary is identical, only the transport changes:

```python
import requests

r = requests.post("http://127.0.0.1:5001/predict/price", json=car, timeout=5)
r.raise_for_status()
print(r.json())          # {'price_lakhs': 4.87}
```

`raise_for_status()` is the line that separates a client from a script: it turns a `4xx`/`5xx`
into a Python exception instead of letting `r.json()` fail confusingly two lines later.

That cell is not executed here, because it needs a process listening on port 5001.

## 3.4 The service you started, from the command line

This is the process from §3.0 — either spelling of the command starts it:

```bash
flask --app flask_app.py run --port 5001
python -m flask --app flask_app.py run --port 5001
```

`curl` is the other way to reach it, from a *second* terminal:

```bash
curl --location 'http://127.0.0.1:5001/monday'

curl --location 'http://127.0.0.1:5001/predict/price' \
  --header 'Content-Type: application/json' \
  --data '{
    "year": 2018,
    "seller_type": "Dealer",
    "km_driven": 45000,
    "fuel_type": "Petrol",
    "transmission_type": "Manual",
    "mileage": 18.5,
    "engine": 1200,
    "max_power": 85.0,
    "seats": 5
  }'

curl --location 'http://127.0.0.1:5001/predict/loan' \
  --header 'Content-Type: application/json' \
  --data '{
    "Gender": "Male",
    "Married": "Unmarried",
    "Credit_History": "Cleared Debts",
    "ApplicantIncome": 50000,
    "LoanAmount": 5
  }'
```

`Ctrl-C` in the first terminal stops it. Note the banner Flask prints on startup:
**"WARNING: This is a development server. Do not use it in a production deployment."** P8 takes
that warning seriously.

Note also `"Married": "Unmarried"` in that last payload — a value the training data never
contained. It returns `200`. Hold that thought for exactly one Part.

```
P0 setup  P1 models  P2 the web  P3 flask  [ P4 FLASK'S GAPS ]  P5 fastapi  P6 the seam  P7 async  P8 workers  P9 pick one
```

# Part 4 — What Flask does not do for you

The service works for well-formed requests. Every payload below is one the partner bank will
send you in the first month, and Flask handles all of them the same way: it doesn't.

The first two are loud. The next two are not, and they are the ones that cost money.

## 4.1 A missing field

In [19]:
incomplete = {
    "Gender": "Male",
    "Married": "No",
    "ApplicantIncome": 5000,
    "LoanAmount": 128,
}   # Credit_History is missing

r = client.post("/predict/loan", json=incomplete)
print("status:", r.status_code, " <- a 5xx: Milepost's fault, says the caller's monitoring")
print(r.get_data(as_text=True)[:300])

[2026-09-23 20:09:08,939] ERROR in app: Exception on /predict/loan [POST]
Traceback (most recent call last):
  File "/Users/shivam13juna/Documents/virtual_envs/dev3.12/lib/python3.12/site-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/shivam13juna/Documents/virtual_envs/dev3.12/lib/python3.12/site-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/shivam13juna/Documents/virtual_envs/dev3.12/lib/python3.12/site-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/shivam13juna/Documents/virtual_envs/dev3.12/lib/python3.12/site-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^

status: 500  <- a 5xx: Milepost's fault, says the caller's monitoring
<!doctype html>
<html lang=en>
<title>500 Internal Server Error</title>
<h1>Internal Server Error</h1>
<p>The server encountered an internal error and was unable to complete your request. Either the server is overloaded or there is an error in the application.</p>

time: 1.26 ms (started: 2026-09-23 20:09:08 +05:30)


A missing key became a `KeyError`, which became an unhandled exception, which became a **500**.

Two different things printed above, and the difference is the whole lesson. The traceback is the
**server log** — it names the file, the line and the key, and only you can see it. The
`<!doctype html>` block is the **response**, which is all the caller gets: a generic 500 page
that does not mention `Credit_History` at all.

Read that from the bank's side. They sent a bad payload, and your service reported that *it* had
failed. Their dashboard shows Milepost's uptime dropping. Your on-call gets paged at 03:00 for a
bug in someone else's code. And to find out which field was wrong, someone has to ask you, and
you have to go and read that traceback.

## 4.2 A wrong type

In [20]:
import logging

# Flask logged the whole traceback for the failure above -- that is what your logs look like.
# Quiet it for the rest of this Part; from here the status code is the point.
flask_app.app.logger.setLevel(logging.CRITICAL)

wrong_types = dict(car, year="two thousand eighteen")

r = client.post("/predict/price", json=wrong_types)
print("status:", r.status_code, " <- 500 again, this time from float('two thousand eighteen')")

status: 500  <- 500 again, this time from float('two thousand eighteen')
time: 491 µs (started: 2026-09-23 20:09:08 +05:30)


In [21]:
unknown_fuel = dict(car, fuel_type="Hydrogen")

r = client.post("/predict/price", json=unknown_fuel)
print("status:", r.status_code, " <- 500 again: encode_dict['fuel_type']['Hydrogen'] is a KeyError")

status: 500  <- 500 again: encode_dict['fuel_type']['Hydrogen'] is a KeyError
time: 346 µs (started: 2026-09-23 20:09:08 +05:30)


Both loud, both the wrong status code, both blaming the wrong party. At least they are visible.

And that second one was luck. `encode_dict["fuel_type"]["Hydrogen"]` raises because a plain
`dict` has no opinion about missing keys — it fails. Had the categorical been encoded with
scikit-learn's `OneHotEncoder(handle_unknown="ignore")`, which is the common choice, `Hydrogen`
would have become a row of zeros and the service would have priced the car at status `200` with
no error anywhere. Same bug, one configuration flag apart, and only the request schema settles
which of the two you get.

Which is the subject of the next two subsections.

## 4.3 The one that does not raise at all

Before the demo, the objection this section has to survive: *in production you do not send
random values.* That is true. Integrations are agreed, the strings are produced by code rather
than typed, and a well-configured caller sends exactly what was specified.

**This subsection is about what happens when the caller does everything right.**

Look at what `predict_loan` does in `flask_app.py`:

```python
if loan_req["Married"] == "Unmarried":
    Married = "No"
else:
    Married = "Yes"
```

Whoever wrote that was expecting the word `"Unmarried"`. **That word does not exist.** The
`Married` column of the training data holds exactly two values, the ones §1.4 encoded —
`{"No": 0, "Yes": 1}` — and `"Unmarried"` appears nowhere: not in the CSV, not in the encoder,
not in any contract anybody agreed.

So a caller who sends `"No"` has sent one of only two legal values, correctly. It is not
`"Unmarried"`, so it falls into the `else`, and an unmarried applicant is filed as **married**.

In [22]:
for married in ("No", "Yes"):
    row = pd.DataFrame(
        [[0, loan_encode_dict["Married"][married], 5000.0, 128.0, 1]], columns=LOAN_FEATURES
    )
    p = float(loan_model.predict_proba(row)[0, 1])
    verdict = "Loan Approved" if p >= DECISION_THRESHOLD else "Loan Rejected"
    print(f"the model, Married={married!r:5} -> probability {p:.4f} -> {verdict}")

sent = client.post("/predict/loan", json=dict(buyer, Married="No")).get_json()
print()
print(f"flask, sent Married='No'    -> {sent['loan_approval_status']}  <- it used the 'Yes' row")

the model, Married='No'  -> probability 0.7448 -> Loan Rejected
the model, Married='Yes' -> probability 0.8359 -> Loan Approved

flask, sent Married='No'    -> Loan Approved  <- it used the 'Yes' row
time: 1.74 ms (started: 2026-09-23 20:09:08 +05:30)


Status `200`. No exception, no warning, no log line, and **nothing upstream is misconfigured.**
The payload was correct, the integration was correct, the caller has nothing to fix. The wrong
value was invented inside the handler, by an `else` that had to guess and guessed.

That is the answer to the objection. The expensive bug here does not need anybody to send
something random — it needs a handler that turns "I did not recognise this" into a default.

## 4.4 The same bug, reached from the other side

The `else` is also waiting for the value that *did* get agreed. Here is that path, and it is the
one the objection applies to — so be specific about how it happens, because nobody hand-types
production payloads:

- a `.lower()` or `.strip().title()` in a form handler between the user and the API
- a CSV or spreadsheet round-trip in an overnight job
- an upstream dropdown renamed — the CRM changes `"Unmarried"` to `"Single"` and the caller faithfully forwards what it is now given
- four independently-deployed callers, one of which is a release behind

None of those are somebody being careless. Each is a value that was right when it left and
arrived changed.

In [23]:
has_debts = dict(buyer, Credit_History="Uncleared Debts")
mangled   = dict(buyer, Credit_History="uncleared debts")   # same buyer, lowercased in transit

for label, payload in [('"Uncleared Debts"', has_debts), ('"uncleared debts"', mangled)]:
    r = client.post("/predict/loan", json=payload)
    print(f"{label:<20} -> {r.status_code} {r.get_json()}")

"Uncleared Debts"    -> 200 {'loan_approval_status': 'Loan Rejected', 'model_version': 'milepost-2026-09-23', 'probability': 0.1906, 'threshold_applied': 0.8}
"uncleared debts"    -> 200 {'loan_approval_status': 'Loan Approved', 'model_version': 'milepost-2026-09-23', 'probability': 0.8359, 'threshold_applied': 0.8}
time: 1.45 ms (started: 2026-09-23 20:09:08 +05:30)


The same buyer, with uncleared debts, approved for finance — because one character of casing
changed somewhere between the form and here.

```python
if loan_req["Credit_History"] == "Uncleared Debts":
    Credit_History = "Uncleared Debts"
else:
    Credit_History = "Cleared Debts"
```

Any value that is not that exact string is filed as **cleared debts**: not rejected, not logged,
defaulted to the more generous answer. And §1.4 measured why that is the field to care about —
`Credit_History` is what decides the lending call, and it is what silently defaults.

This is the expensive failure mode, and it is not the crash. A `500` gets noticed in an
afternoon. This one ships, runs for a quarter, and is found by the credit losses.

`Gender` has the same `else`, and defaults just as quietly.

### What validation actually costs when the contract holds

Which is worth stating plainly, because the objection at the top of this section is usually
raised as a reason *not* to validate:

> If your callers really are well configured, a request schema **never fires**. Every request
> passes, every time, and it has cost you nothing. It only ever fires when something upstream
> has already broken — a rename, a transform, a stale deploy.

So "our integration is agreed" is an argument *for* putting the contract in the code, not
against it. It was never a choice between a `422` and a clean life. It is a choice between a
`422` and an `else` that guesses.

## 4.5 You can write the validation yourself

Nothing stops you. Here is the honest minimum for *one* endpoint with *five* fields.

In [24]:
ALLOWED = {
    "Gender": {"Male", "Female"},
    "Married": {"Yes", "No"},
    "Credit_History": {"Cleared Debts", "Uncleared Debts"},
}
NUMERIC = ["ApplicantIncome", "LoanAmount"]


def validate_loan(payload):
    """Return a list of problems with a /predict/loan body. Empty list means valid."""
    problems = []

    for field, allowed in ALLOWED.items():
        if field not in payload:
            problems.append(f"{field}: missing")
        elif payload[field] not in allowed:
            problems.append(f"{field}: {payload[field]!r} not in {sorted(allowed)}")

    for field in NUMERIC:
        if field not in payload:
            problems.append(f"{field}: missing")
        else:
            try:
                if float(payload[field]) < 0:
                    problems.append(f"{field}: must be >= 0")
            except (TypeError, ValueError):
                problems.append(f"{field}: {payload[field]!r} is not a number")

    return problems


for label, payload in [("valid", buyer), ("incomplete", incomplete), ("mangled", mangled)]:
    print(f"{label:<12} -> {validate_loan(payload) or 'ok'}")

valid        -> ok
incomplete   -> ['Credit_History: missing']
mangled      -> ["Credit_History: 'uncleared debts' not in ['Cleared Debts', 'Uncleared Debts']"]
time: 584 µs (started: 2026-09-23 20:09:08 +05:30)


It works. It is also 25 lines that catch exactly one endpoint's mistakes, it has to be written
again for `/predict/price` and its nine fields, it has to be kept in step with the model every
time a feature is added, it produces error messages only you will ever understand, and it
documents nothing for the caller.

Now count what is missing from that list even so: it does not coerce `"45000"` into a number, it
does not appear in any documentation, and a new endpoint starts from zero.

Everything Flask is missing in this Part is the same missing thing: **a declaration of what a
valid request looks like.** Flask hands you `request.get_json()` — a plain `dict` — and wishes
you luck. You can write the validation by hand, for every field, on every endpoint, forever. Or
you can write down the types and let the framework do it.

```
P0 setup  P1 models  P2 the web  P3 flask  P4 flask's gaps  [ P5 FASTAPI ]  P6 the seam  P7 async  P8 workers  P9 pick one
```

# Part 5 — FastAPI: the same API, with a contract

FastAPI's idea is small and it changes everything: **declare the request as a Python type, and
the framework enforces it before your function runs.**

```python
class LoanRequest(BaseModel):
    Gender: Literal["Male", "Female"]
    ApplicantIncome: float = Field(ge=0)

@app.post("/predict/loan")
def predict_loan(req: LoanRequest):   # <- if the body does not match, this never executes
    ...
```

From that one annotation FastAPI derives, for free:

| | |
|---|---|
| **Validation** | Wrong type, missing field, value out of range → `422` with the exact path to the problem |
| **Parsing** | `"45000"` becomes `45000.0`; your function receives typed Python, not a `dict` |
| **Documentation** | An **OpenAPI** description of the service at `/openapi.json`, and a clickable UI at `/docs` that draws it |
| **Editor support** | `req.` autocompletes, because it is a real class |

`Literal[...]` is the direct fix for §4.4's silent approval — a value outside the list is
rejected rather than quietly bucketed into an `else`.

## 5.0 Start it first, in a terminal you own

Same arrangement as Part 3, different command. The cells below need nothing listening; Postman
does. Start it before reading the file:

```bash
uvicorn fastapi_app:app --reload --port 8001
```

`fastapi_app:app` means *the object named `app` inside `fastapi_app.py`*. `--reload` restarts on
every file save — convenient while writing, never in production. And `--port` for the same reason
as last Part: uvicorn's default is **8000**, the most frequently occupied port on any development
machine.

It serves on `http://127.0.0.1:8001`. There is also a clickable UI at
**`http://127.0.0.1:8001/docs`**, generated without anyone writing it — §5.7 comes back to where
it came from.

**In Postman**, identical to Part 3: `POST`, **raw → JSON**, paste the body, Send. Point it at
`http://127.0.0.1:8001` instead of `5001` and send the *same* payloads. The difference you are
looking for is in the reply — where Flask gave you a `500` and an HTML page, this gives you a
`422` and a JSON body that names the field that was wrong, in Postman, without reading the
server's terminal.

## 5.1 The file

Open **`fastapi_app.py`**. The model layer at the top is a deliberate copy of the one in
`flask_app.py` — same dictionaries, same two loads in the module body, same two prediction
functions. One file you can read end to end beats a file plus a private module you have to go
and find, and P9 comes back to what that duplication costs.

What is *not* a copy is the middle of the file:

```python
class PriceRequest(BaseModel):
    year: int = Field(ge=1991, le=2021, description="Manufacturing year")
    seller_type: Literal["Dealer", "Individual", "Trustmark Dealer"]
    km_driven: float = Field(ge=0, le=400_000)
    fuel_type: Literal["Diesel", "Petrol", "CNG", "LPG", "Electric"]
    ...
```

Nine lines that say what a car is. Every one of them is a sentence about the world — a year
before 1991 is not in this dataset, a negative odometer is not a car — and the framework now
enforces all nine on every request forever.

## 5.2 Calling it without starting a server

Same in-process idea as Flask, spelled `TestClient`. Identical routing, identical validation,
no port.

In [25]:
from fastapi.testclient import TestClient

import fastapi_app

api = TestClient(fastapi_app.app)

print("GET /health          ->", api.get("/health").status_code, api.get("/health").json())
print("POST /predict/price  ->", api.post("/predict/price", json=car).json())
print("POST /predict/loan   ->", api.post("/predict/loan", json=buyer).json())

GET /health          -> 200 {'status': 'ok', 'models': ['price', 'loan'], 'canary_price_lakhs': 5.31, 'decision_threshold': 0.8, 'model_version': 'milepost-2026-09-23', 'sklearn_version': '1.7.2', 'sklearn_version_at_train': '1.7.2'}
POST /predict/price  -> {'price_lakhs': 5.31, 'model_version': 'milepost-2026-09-23'}
POST /predict/loan   -> {'loan_approval_status': 'Loan Rejected', 'probability': 0.7448, 'threshold_applied': 0.8, 'model_version': 'milepost-2026-09-23'}
time: 35 ms (started: 2026-09-23 20:09:08 +05:30)


Same two models, same two numbers as Flask — the model layer is unchanged, so it had better be.
Now send the three payloads that broke it.

In [26]:
r = api.post("/predict/loan", json=incomplete)   # Credit_History missing
print("status:", r.status_code, " <- 4xx: the caller's fault, and it says so")
print(json.dumps(r.json(), indent=2))

status: 422  <- 4xx: the caller's fault, and it says so
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "Credit_History"
      ],
      "msg": "Field required",
      "input": {
        "Gender": "Male",
        "Married": "No",
        "ApplicantIncome": 5000,
        "LoanAmount": 128
      }
    }
  ]
}
time: 1.12 ms (started: 2026-09-23 20:09:08 +05:30)


`422 Unprocessable Entity`. The body names the field (`loc`), the problem (`missing`), and a
human-readable message. The bank can fix their payload without emailing anyone, your error rate
is unaffected, and nobody gets paged.

In [27]:
r = api.post("/predict/price", json=wrong_types)   # year="two thousand eighteen"
print("status:", r.status_code)
print(json.dumps(r.json()["detail"], indent=2))

status: 422
[
  {
    "type": "int_parsing",
    "loc": [
      "body",
      "year"
    ],
    "msg": "Input should be a valid integer, unable to parse string as an integer",
    "input": "two thousand eighteen"
  }
]
time: 1.19 ms (started: 2026-09-23 20:09:09 +05:30)


In [28]:
r = api.post("/predict/price", json=unknown_fuel)   # fuel_type="Hydrogen"
print("status:", r.status_code)
print(json.dumps(r.json()["detail"], indent=2))

status: 422
[
  {
    "type": "literal_error",
    "loc": [
      "body",
      "fuel_type"
    ],
    "msg": "Input should be 'Diesel', 'Petrol', 'CNG', 'LPG' or 'Electric'",
    "input": "Hydrogen",
    "ctx": {
      "expected": "'Diesel', 'Petrol', 'CNG', 'LPG' or 'Electric'"
    }
  }
]
time: 1.02 ms (started: 2026-09-23 20:09:09 +05:30)


The `KeyError` from §4.2 is now a sentence telling the caller which five strings are acceptable.
Note where that list came from: nobody wrote documentation. It is the `Literal` in the class,
read back.

And the one that matters:

In [29]:
r = api.post("/predict/loan", json=mangled)   # Credit_History="uncleared debts"
print("status:", r.status_code, " <- the silent approval from §4.4, now impossible")
print(json.dumps(r.json()["detail"], indent=2))

status: 422  <- the silent approval from §4.4, now impossible
[
  {
    "type": "literal_error",
    "loc": [
      "body",
      "Credit_History"
    ],
    "msg": "Input should be 'Cleared Debts' or 'Uncleared Debts'",
    "input": "uncleared debts",
    "ctx": {
      "expected": "'Cleared Debts' or 'Uncleared Debts'"
    }
  }
]
time: 1.03 ms (started: 2026-09-23 20:09:09 +05:30)


The value that Flask bucketed as *cleared debts* — approving a buyer who should have been
rejected, at status `200` — is now refused, with a list of exactly what is allowed.

The bug was not fixed by being more careful. It was fixed by being **declared**.
`Literal["Cleared Debts", "Uncleared Debts"]` is a statement about the world, and there is no
`else` branch for the framework to fall into.

## 5.3 Strict is not the same as pedantic

A reasonable worry about all this: will it reject the bank's perfectly sensible request because
a number arrived as a string? No — coercion is part of the declaration.

In [30]:
r = api.post("/predict/price", json=dict(car, km_driven="45000", year="2018"))
print(r.status_code, r.json(), " <- strings coerced to numbers, because the type says so")

r = api.post("/predict/price", json=dict(car, km_driven=-5))
print(r.status_code, r.json()["detail"][0]["msg"], " <- Field(ge=0) enforced")

r = api.post("/predict/price", json=dict(car, colour="red"))
print(r.status_code, r.json(), " <- an extra field it does not know about is ignored, not fatal")

200 {'price_lakhs': 5.31, 'model_version': 'milepost-2026-09-23'}  <- strings coerced to numbers, because the type says so
422 Input should be greater than or equal to 0  <- Field(ge=0) enforced
200 {'price_lakhs': 5.31, 'model_version': 'milepost-2026-09-23'}  <- an extra field it does not know about is ignored, not fatal
time: 6.48 ms (started: 2026-09-23 20:09:09 +05:30)


Three sensible behaviours nobody implemented: convert what is convertible, refuse what is out of
range, ignore what is irrelevant.

## 5.4 The contract, written down for you

Those nine lines are not only enforced. They are **published**.

**OpenAPI** is a standard for describing an HTTP API as a JSON document: the routes, the shape of
every request and response, which fields are required, which values they accept. It is a format
rather than a library, so the same document can describe a Java or a Go service — which is why so
much tooling can read it. Its earlier name was Swagger, and that name survives in the tools built
around it.

FastAPI assembles that document out of the type annotations and serves it at **`/openapi.json`**.
`app.openapi()` hands you the same document as a Python `dict`, which is why it prints here with
no server running. Two keys carry the contract: `paths`, every route and method, and
`components.schemas`, one entry per model.

Nobody maintains this file, so it cannot go stale — and it is what a client generator, a test
tool, or the bank's integration team reads instead of asking you.

In [31]:
schema = fastapi_app.app.openapi()
print("openapi version:", schema["openapi"])
print("top-level keys :", list(schema))
print()
print(json.dumps(schema["components"]["schemas"]["LoanRequest"], indent=2))

openapi version: 3.1.0
top-level keys : ['openapi', 'info', 'paths', 'components']

{
  "properties": {
    "Gender": {
      "type": "string",
      "enum": [
        "Male",
        "Female"
      ],
      "title": "Gender"
    },
    "Married": {
      "type": "string",
      "enum": [
        "Yes",
        "No"
      ],
      "title": "Married"
    },
    "ApplicantIncome": {
      "type": "number",
      "minimum": 0.0,
      "title": "Applicantincome",
      "description": "Monthly income"
    },
    "LoanAmount": {
      "type": "number",
      "minimum": 0.0,
      "title": "Loanamount",
      "description": "Requested amount, in thousands"
    },
    "Credit_History": {
      "type": "string",
      "enum": [
        "Cleared Debts",
        "Uncleared Debts"
      ],
      "title": "Credit History"
    }
  },
  "type": "object",
  "required": [
    "Gender",
    "Married",
    "ApplicantIncome",
    "LoanAmount",
    "Credit_History"
  ],
  "title": "LoanRequest"
}
time: 6.9

In [32]:
print("documented endpoints:")
for path, methods in schema["paths"].items():
    for method, spec in methods.items():
        print(f"  {method.upper():<5} {path:<18} {spec.get('summary', '')}")

documented endpoints:
  GET   /health            Health
  POST  /predict/price     Predict Price
  POST  /predict/price/batch Predict Price Batch
  POST  /predict/loan      Predict Loan
time: 178 µs (started: 2026-09-23 20:09:09 +05:30)


`enum` in that schema is the `Literal`. `required` is the list of fields with no default.
`minimum` is `Field(ge=0)`. Nothing was written twice.

## 5.5 Many cars, one round trip

The bank's loan desk asks about one buyer at a time. The overnight repricing job does not — it
wants the whole forecourt. There are two ways to give it one: call `/predict/price` in a loop, or
call `/predict/price/batch` once.

Both services have the batch route, and they are written differently on purpose.

```python
# flask_app.py -- the obvious way, and what gets written first in both frameworks
cars = request.get_json()
return {"prices_lakhs": [model_pred(c["year"], ...) for c in cars],
        "model_version": MODEL_VERSION}

# fastapi_app.py
@app.post("/predict/price/batch", response_model=list[PriceResponse])
def predict_price_batch(cars: list[PriceRequest]):
    if not cars:
        raise HTTPException(status_code=422, detail="send at least one car")
    if len(cars) > 1000:
        raise HTTPException(status_code=413, detail="at most 1000 cars per call")
    frame = pd.DataFrame([[...] for c in cars], columns=PRICE_FEATURES)
    return [PriceResponse(price_lakhs=round(float(p), 2), model_version=MODEL_VERSION)
            for p in price_model.predict(frame)]
```

Two separate things are going on there, and it is worth keeping them apart.

**`list[PriceRequest]` is the framework doing work.** One annotation, and every element of the
body is validated; one bad car and the whole request is refused before any of it runs. Those two
`raise` lines are the only hand-written error handling in the file — a batch endpoint with no
upper bound is how a caller turns one HTTP request into an out-of-memory kill.

**One `predict` call over a frame is not a FastAPI feature.** Either framework could do it, and
Flask's loop here is the version people write first. It is also where the time goes.

In [33]:
fleet = [dict(car, km_driven=40000 + 500 * i) for i in range(100)]

t0 = time.perf_counter()
one_by_one = [api.post("/predict/price", json=c).json()["price_lakhs"] for c in fleet]
loop_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
batched = [r["price_lakhs"] for r in api.post("/predict/price/batch", json=fleet).json()]
batch_ms = (time.perf_counter() - t0) * 1000

print(f"100 cars, one request each : {loop_ms:7.1f} ms")
print(f"100 cars, one batch call   : {batch_ms:7.1f} ms   ({loop_ms / batch_ms:.0f}x faster)")
print(f"same prices either way     : {one_by_one == batched}")

100 cars, one request each :   171.3 ms
100 cars, one batch call   :     2.8 ms   (62x faster)
same prices either way     : True
time: 174 ms (started: 2026-09-23 20:09:09 +05:30)


The last line is the one that matters — a batch endpoint that is faster and disagrees with the
single endpoint is not an optimisation, it is a second implementation waiting to drift. Same
numbers, or it does not ship.

And the guards do what they say:

In [34]:
print("empty list        ->", api.post("/predict/price/batch", json=[]).status_code,
      api.post("/predict/price/batch", json=[]).json()["detail"])

too_many = api.post("/predict/price/batch", json=[car] * 1001)
print("1001 cars         ->", too_many.status_code, too_many.json()["detail"])

one_rotten = api.post("/predict/price/batch", json=[car, dict(car, fuel_type="Hydrogen"), car])
print("one bad element   ->", one_rotten.status_code,
      "· refused at index", one_rotten.json()["detail"][0]["loc"][1])

empty list        -> 422 send at least one car


1001 cars         -> 413 at most 1000 cars per call
one bad element   -> 422 · refused at index 1
time: 6.76 ms (started: 2026-09-23 20:09:09 +05:30)


`422`, `413`, and a `422` that names **which car in the list** was wrong. Send that same
three-car list to the Flask route and it prices the first car, raises on the second, and returns
a `500` — having already done a third of the work and telling the caller nothing about where it
stopped.

## 5.6 Who owns the decision

`/predict/loan` returns four fields, and only one of them is the answer.

```python
class LoanResponse(BaseModel):
    loan_approval_status: str    # the decision this service applied
    probability: float           # the basis for it
    threshold_applied: float     # the cut that turned one into the other
    model_version: str           # which artifact produced the probability
```

The narrow reason is that without `threshold_applied` a wrong decision is unfalsifiable. The
wider one is that **the decision and the model change on completely different clocks**, and the
response shape is what decides whether that costs you a deployment.

### The number cannot come from the caller

The first instinct is to accept it in the payload — `{"threshold": 0.8, ...}` — and it is worth
being specific about why that is not done. Whoever sends the threshold *is* the lending policy.
One API key and a `0.01` approves everybody, with a `200` and clean logs. And a caller could not
choose well even in good faith: `0.80` is only meaningful against this model version's
probability scale, which the caller knows nothing about.

Where a threshold legitimately appears as a parameter: a `/score?threshold=` endpoint behind
auth, for analysts sweeping cuts. And in multi-tenant serving — but there the caller sends
`tenant_id` and the **server** looks up that tenant's cut. The caller identifies itself; it does
not set policy.

### Four places it can live

| | where | changing it costs | the risk |
|---|---|---|---|
| 1 | inside the artifact — `FixedThresholdClassifier(model, threshold=0.80)` pickles the cut into the `.joblib`, so `predict()` *means* 0.80 | a retrain-and-repackage release | invisible: you cannot read it without unpickling, and loading needs the same sklearn |
| 2 | **beside the artifact** — `model_meta.json`, what this notebook ships | a model release, but the file is plain text | none, but it is coupled to model releases |
| 3 | deploy config — env var, ConfigMap, Helm value | a restart | drifts from the model: recalibrate and the inherited `0.80` silently means a different approval rate |
| 4 | a decision service that owns lending policy outright | a policy change, no deploy | a dependency near the hot path, so cache it and log which policy version scored each request |

Option 1 is worth knowing about precisely because it contradicts the usual claim that a model
"cannot carry" its threshold. It can. It is just rarely where you want it — unless the artifact
is your *only* delivery channel, or the cut was genuinely fitted, by something like
`TunedThresholdClassifierCV`, in which case it really is part of the estimator.

Option 3 is what most single teams do, and it is the one to be careful with: probability scales
are not portable across model versions, so a threshold in config has to be re-checked against
every model it is paired with.

### At scale, the model service does not decide

In a bank or a large lender the cut lives in a **decision engine** — its own system, its own
repo, its own reviewers. The model is one input to it, next to bureau data, KYC flags and
exposure limits. Four things push it out of the model service and none of them are preferences:

- **Regulation.** An adverse-action notice has to state the basis of a decline, and auditors want the policy change log separate from model lineage. "It is inside the pickle" does not survive an examination.
- **Ownership.** A credit policy team changes cuts. They are not going to open a pull request against a serving repo.
- **Multi-model reality.** One decision consumes several scores. There is no single model to attach the number to.
- **Segmentation.** Different cuts per region, product and experiment arm. That is a lookup, not a constant.

### Which is why the response carries both

Milepost has no decision engine, and its callers are a browser, a `curl` and an overnight job.
If the service returned only a probability, the cut would end up hardcoded in each of them — and
"Milepost's lending policy" would be three numbers that nobody can state.

So the service decides, using a cut it read from a file, and hands back the basis anyway:

In [35]:
# Married="Yes" so that §4.3's Flask `else` does not muddy this -- the point here is
# the shape of the response, not the defect behind it. This buyer sits between the two
# cuts: 0.80 rejects, the library's 0.5 approves.
marginal = dict(buyer, Married="Yes", LoanAmount=400)

print("flask   :", client.post("/predict/loan", json=marginal).get_json())
print("fastapi :", api.post("/predict/loan", json=marginal).json())

# Now the failure the fourth field exists to expose. Both services read the cut from
# model_meta.json, so this has to be simulated: forcing THRESHOLD to 0.5 is exactly what
# a handler reaching for `loan_model.predict()` would have done. Put back on the next line.
saved = flask_app.THRESHOLD
flask_app.THRESHOLD = 0.5
print()
print("predict():", client.post("/predict/loan", json=marginal).get_json())
flask_app.THRESHOLD = saved

flask   : {'loan_approval_status': 'Loan Rejected', 'model_version': 'milepost-2026-09-23', 'probability': 0.7578, 'threshold_applied': 0.8}
fastapi : {'loan_approval_status': 'Loan Rejected', 'probability': 0.7578, 'threshold_applied': 0.8, 'model_version': 'milepost-2026-09-23'}

predict(): {'loan_approval_status': 'Loan Approved', 'model_version': 'milepost-2026-09-23', 'probability': 0.7578, 'threshold_applied': 0.5}
time: 3.46 ms (started: 2026-09-23 20:09:09 +05:30)


Read the third line against the first. **Same buyer, same payload, the same probability — and
the opposite verdict.** Status `200` both times, no exception, no warning, no log line, and
nothing malformed to find in a request log afterwards. Every field the caller sent was perfect.

The only evidence that anything changed is `threshold_applied` going from `0.8` to `0.5`. That
is the field earning its place. Take it away and a response of
`{"loan_approval_status": "Loan Approved"}` is unfalsifiable — it cannot be checked, replayed, or
explained to anyone asking six months later why this applicant was approved. `probability`
answers *how close was it*, and `model_version` answers *which model said so*.

It also says why the cut is worth reading from a file rather than typing into the handler.
`loan_model.predict(frame)` is the obvious call, it reads like a decision, and it is one — just
not yours. It means 0.5, silently, and overrides whatever the business agreed.

That shape is also what survives growing up. The day a real decision layer appears it ignores
`loan_approval_status`, reads `probability`, and applies its own policy — a change on their side,
with no new version of this API. Ship the probability alone and you can never add a server-side
decision without ambiguity about who decides; ship the decision alone and the caller can never
take ownership of it.

One thing the table above does not include, because it is not a choice: **there is no fallback.**
Both services read `META["decision_threshold"]` at import, and a missing key raises there, before
a single request is served. Part 8 shows what that looks like from outside the process.

## 5.7 The free UI

The process from §5.0 is still the one serving:

```bash
uvicorn fastapi_app:app --reload --port 8001
```

Two URLs on it are worth opening, and neither of them was written by anyone:

- **`http://127.0.0.1:8001/docs`** — Swagger UI: §5.4's `/openapi.json` document, drawn. Every endpoint has a **Try it out** button that sends a real request. This is usually the moment FastAPI sells itself: the bank can integrate without you writing a single page of documentation.
- **`http://127.0.0.1:8001/redoc`** — the same document, a different renderer, laid out as a reference page.

And the same `curl` as P3, on the new port:

```bash
curl --location 'http://127.0.0.1:8001/predict/price' \
  --header 'Content-Type: application/json' \
  --data '{
    "year": 2018,
    "seller_type": "Dealer",
    "km_driven": 45000,
    "fuel_type": "Petrol",
    "transmission_type": "Manual",
    "mileage": 18.5,
    "engine": 1200,
    "max_power": 85.0,
    "seats": 5
  }'
```

Try the lowercase `"uncleared debts"` in `/docs` once. The UI will not even let you type it —
the field is a dropdown, because the schema says it is one of two values.

```
P0 setup  P1 models  P2 the web  P3 flask  P4 flask's gaps  P5 fastapi  [ P6 THE SEAM ]  P7 async  P8 workers  P9 pick one
```

# Part 6 — The seam, and the one test that watches it

Everything so far has been about the request. This Part is about the other side of the handler,
where the service turns validated fields into the row the model was trained on. That is **the
seam**, and it is the part of a service that breaks without moving any score.

Three things have to agree at the seam, and nothing in Python checks any of them:

| | Training built it | Serving builds it |
|---|---|---|
| The encoding | `df.replace(encode_dict)` over a whole frame | `encode_dict[...]` per field, in a service file |
| The column order | `df[PRICE_FEATURES]` | a list literal, typed out again |
| The threshold | not at all — it did not exist yet | `predict_proba(...) >= THRESHOLD` |

Our situation is worse than the usual one, deliberately: there are **three** copies of that
encoding now — the notebook's, `flask_app.py`'s and `fastapi_app.py`'s — because §5.1 chose
readable files over a shared module. So the question is not whether the copies could drift. It
is what notices when they do.

## 6.1 Score the same car twice

The test is six lines and it is the only one worth copying into your own projects. Take one car.
Score it through HTTP, with the JSON parsing and the validation and the frame building in
between. Score it again by calling the model directly, the way Part 1 did. Require the two
numbers to match.

In [36]:
def training_path(c):
    """The row the way Part 1 built rows -- the training side of the seam."""
    return pd.DataFrame(
        [[
            float(c["year"]),
            encode_dict["seller_type"][c["seller_type"]],
            float(c["km_driven"]),
            encode_dict["fuel_type"][c["fuel_type"]],
            encode_dict["transmission_type"][c["transmission_type"]],
            float(c["mileage"]),
            float(c["engine"]),
            float(c["max_power"]),
            float(c["seats"]),
        ]],
        columns=PRICE_FEATURES,
    )


direct = round(float(price_model.predict(training_path(car))[0]), 2)
via_flask = client.post("/predict/price", json=car).get_json()["price_lakhs"]
via_fastapi = api.post("/predict/price", json=car).json()["price_lakhs"]

print(f"the model, called directly : {direct}")
print(f"through flask_app.py       : {via_flask}")
print(f"through fastapi_app.py     : {via_fastapi}")
print()
print("all three agree            :", direct == via_flask == via_fastapi)

the model, called directly : 5.31
through flask_app.py       : 5.31
through fastapi_app.py     : 5.31

all three agree            : True
time: 4.73 ms (started: 2026-09-23 20:09:09 +05:30)


A serving layer that reorders a column, encodes a category differently, or rounds somewhere
unexpected passes every other check in this notebook — the status code is still `200`, the
response still has the right shape, `/docs` still renders — and fails this one.

The price endpoint holds.

## 6.2 The same test, on the other endpoint

Identical shape, nothing new to explain — build the row the way Part 1 built rows, apply the
stored cut, and compare that against what each service answers.

In [37]:
def loan_training_path(b):
    """The row the way Part 1 built rows, plus §1.5's threshold."""
    row = pd.DataFrame(
        [[
            loan_encode_dict["Gender"][b["Gender"]],
            loan_encode_dict["Married"][b["Married"]],
            float(b["ApplicantIncome"]),
            float(b["LoanAmount"]),
            loan_encode_dict["Credit_History"][b["Credit_History"]],
        ]],
        columns=LOAN_FEATURES,
    )
    p = float(loan_model.predict_proba(row)[0, 1])
    return "Loan Approved" if p >= DECISION_THRESHOLD else "Loan Rejected", p


expected, probability = loan_training_path(buyer)
f = client.post("/predict/loan", json=buyer).get_json()
a = api.post("/predict/loan", json=buyer).json()

print(f"the model, called directly : {expected:<14} probability {probability:.4f}")
print(f"through flask_app.py       : {f['loan_approval_status']:<14} probability {f['probability']:.4f}")
print(f"through fastapi_app.py     : {a['loan_approval_status']:<14} probability {a['probability']:.4f}")
print()
# Compare like with like: both services round the probability to 4 places before
# putting it in the response, so round the model's own number the same way.
expected_p = round(probability, 4)

print("verdicts agree             :", expected == f["loan_approval_status"] == a["loan_approval_status"])
print("flask probability matches  :", f["probability"] == expected_p)
print("fastapi probability matches:", a["probability"] == expected_p)
print("same cut applied           :", f["threshold_applied"] == a["threshold_applied"] == DECISION_THRESHOLD)

the model, called directly : Loan Rejected  probability 0.7448
through flask_app.py       : Loan Approved  probability 0.8359
through fastapi_app.py     : Loan Rejected  probability 0.7448

verdicts agree             : False
flask probability matches  : False
fastapi probability matches: True
same cut applied           : True
time: 3.37 ms (started: 2026-09-23 20:09:09 +05:30)


**They do not agree.** Same payload, same buyer, same two models on disk, two services — and one
approves finance while the other refuses it.

This is not a bug planted for the exercise. It is §4.3's `else`, still sitting in
`flask_app.py`, doing what it was always doing: the buyer sent `"Married": "No"`, the handler
did not recognise that string, and it filed them as married. `fastapi_app.py` has no `else` to
fall into because `Literal["Yes", "No"]` made one unnecessary.

Six lines of test, one endpoint compared two ways, and it found a live defect that four Parts of
prose had only described. That is the whole argument for writing it.

And notice which lines localised it. "Verdicts agree: False" says only that something is wrong.
The next three say where:

- FastAPI's probability matches the model exactly, so that path is clean.
- Flask's does not — and not by a rounding margin, by a different number entirely. So the defect is **upstream of the cut**, in how the row was built, not in how it was compared.
- Both services applied the same cut, which rules the configuration out in the same glance.

A test that compared only the two verdict strings would have found the disagreement and left you
guessing which of four Parts caused it. This is the whole reason `probability` and
`threshold_applied` are in the response: they make a service **debuggable from its own output**.

Note what would *not* have found it. Both services return `200`. Both return the documented
response shape. Both pass every status-code check in P3 and P5. `/docs` renders correctly. The
only thing that disagrees is the answer, and the only test that looks at the answer is this one.

## 6.3 Watch it catch something you introduce

A passing test proves nothing until you have seen it fail. So break one copy of the encoding the
way it actually gets broken: not by deleting anything, just by swapping two values. Somebody
reorders a dictionary while tidying up.

In [38]:
saved = flask_app.encode_dict["fuel_type"].copy()

# Petrol and Diesel swapped in ONE of the three copies. Nothing else touched.
flask_app.encode_dict["fuel_type"] = {"Diesel": 2, "Petrol": 1, "CNG": 3, "LPG": 4, "Electric": 5}

r = client.post("/predict/price", json=car)
drifted = r.get_json()["price_lakhs"]

print(f"status                     : {r.status_code}")
print(f"the model, called directly : {direct}")
print(f"through flask_app.py       : {drifted}")
print(f"the seam holds             : {direct == drifted}")
print()
print(f"every petrol car is now priced as a diesel, off by {abs(drifted - direct):.2f} lakhs")

flask_app.encode_dict["fuel_type"] = saved
print(f"restored                   : {client.post('/predict/price', json=car).get_json()['price_lakhs'] == direct}")

status                     : 200
the model, called directly : 5.31
through flask_app.py       : 6.11
the seam holds             : False

every petrol car is now priced as a diesel, off by 0.80 lakhs
restored                   : True
time: 2.54 ms (started: 2026-09-23 20:09:09 +05:30)


Status `200`. Valid JSON. A plausible number. Every field the caller sent was correct, the
schema was satisfied, `/health` would still report `ok`, and the price is wrong on every petrol
car the service sees.

That is what the seam failing looks like, and it is why the comparison in §6.1 is the test that
earns its place. It is also the honest price of the duplication: three copies of `encode_dict`
is three chances to do this, and the only defence is a test that scores the same car both ways.

`_verify_apps.py` beside this notebook runs exactly that check, plus a parse of both files
asserting neither imports the other. Running it is the last thing to do before starting a real
server:

```bash
python _verify_apps.py
```

```
P0 setup  P1 models  P2 the web  P3 flask  P4 flask's gaps  P5 fastapi  P6 the seam  [ P7 ASYNC ]  P8 workers  P9 pick one
```

# Part 7 — `async`, honestly

FastAPI is built on ASGI, so a handler *can* be `async def`. Before the rule for when it should
be, the mechanism — because the common assumption is that `async def` is what lets a service
handle more than one request at a time, and it is not.

There is one **event loop**: a single thread that receives every request. What happens next
depends entirely on how you declared the handler.

| You wrote | Who runs your function | The event loop, meanwhile |
|---|---|---|
| `def` | **a worker thread**, borrowed from a pool | goes straight back to accepting other requests |
| `async def` | **the event loop itself** | nothing else, until your code reaches an `await` |

So a plain `def` is *already* concurrent. FastAPI assumes it might block, keeps it away from the
loop, and gives it a thread. Declaring `async def` does not add concurrency — it moves your code
**onto** the loop, where anything that blocks blocks everybody.

Which makes the rule narrower than the enthusiasm suggests:

| Your handler | Declare it | Why |
|---|---|---|
| Calls a model, does maths, blocks the CPU | `def` | FastAPI runs it in a threadpool, so it cannot block the event loop |
| `await`s a network call, a database, another API | `async def` | The event loop serves other requests while it waits |
| `async def` **containing a blocking call** | — | The worst case: the event loop stalls for every user |

`model_pred` is CPU work with no `await` in it, so **every handler in `fastapi_app.py` is a plain
`def`** — the word `async` appears in that file exactly once, in a comment explaining why it is
not there. Writing `async def` instead would not make a single prediction faster: it would move
the work onto the event loop and make every *other* request wait for it.

The one line to keep from all of this: **the process is asynchronous; your code is not.** Uvicorn
runs the loop whatever you write. You get the concurrency without writing a coroutine, and
without the third row of that table.

Which raises the fair question of what is left of the difference. With every handler a plain
`def`, the two services end up doing the same thing: a thread runs your function start to finish
while the interpreter serialises the Python. That is also what a threaded WSGI server does.

The event loop's advantage is in the part that is *not* your handler. It reads sockets and writes
responses without tying up a thread, so a WSGI server holds a thread for the whole connection
while an ASGI server holds one only while your handler runs — ten thousand mostly-idle
connections cost one loop instead of ten thousand threads. It is a real difference, and it is not
this workload: the bank sends a short request, waits about a millisecond, and hangs up. Nothing
is held open and nothing waits on anything.

What *can* be measured from here, without a load generator and two production servers, is the
per-request cost of each framework — and the answer is not the one the marketing implies.

## 7.1 What a request actually costs

Below is an in-process comparison. It isolates framework overhead per request and deliberately
does not claim to be a benchmark of either server under load — no sockets, no concurrency, no
production server on either side.

In [39]:
N = 300

t0 = time.perf_counter()
for _ in range(N):
    price_model.predict(pd.DataFrame([[2018.0, 1, 45000.0, 2, 1, 18.5, 1200.0, 85.0, 5.0]], columns=PRICE_FEATURES))
predict_ms = (time.perf_counter() - t0) / N * 1000

t0 = time.perf_counter()
for _ in range(N):
    client.post("/predict/price", json=car)
flask_ms = (time.perf_counter() - t0) / N * 1000

t0 = time.perf_counter()
for _ in range(N):
    api.post("/predict/price", json=car)
fastapi_ms = (time.perf_counter() - t0) / N * 1000

print(f"the prediction itself : {predict_ms:6.2f} ms")
print(f"flask   test_client   : {flask_ms:6.2f} ms/request")
print(f"fastapi TestClient    : {fastapi_ms:6.2f} ms/request  (includes pydantic validation)")

the prediction itself :   0.70 ms
flask   test_client   :   0.87 ms/request
fastapi TestClient    :   1.80 ms/request  (includes pydantic validation)
time: 1.01 s (started: 2026-09-23 20:09:09 +05:30)


Read the three numbers in order, because they do not say what people expect.

The prediction is the largest single cost. Flask adds very little on top of it — subtract and
the routing, the JSON parse and the response are a fraction of a millisecond. **FastAPI is the
slower of the two here, by roughly a factor of two**, and the extra is pydantic doing on every
request exactly what §4.5's hand-written validator did: checking nine fields against nine
declarations.

So the honest summary of this Part is: `async` buys nothing for a tree ensemble, and the
contract is not free — it costs about a millisecond a request.

**Now price the alternative.** §4.4 approved a buyer with uncleared debts because a caller sent
lowercase. One millisecond per request against one silently mis-approved loan is not a trade
worth thinking about for very long.

**The reason to choose FastAPI is the `422` in §5.2 and the `/docs` page in §5.7 — not speed.**
Anyone who tells you FastAPI is faster because it is async is describing a workload that waits
on I/O, which this one does not.

```
P0 setup  P1 models  P2 the web  P3 flask  P4 flask's gaps  P5 fastapi  P6 the seam  P7 async  [ P8 WORKERS ]  P9 pick one
```

# Part 8 — The development server is not a server

Both `flask run` and `uvicorn --reload` print a warning when you start them the easy way, and
both warnings are real.

| | Development | Production |
|---|---|---|
| Flask | `flask run --port 5001` — one process, debugger exposed, no restart on crash | `gunicorn -w 4 flask_app:app` |
| FastAPI | `uvicorn fastapi_app:app --reload --port 8001` — reloader watching the filesystem | `uvicorn fastapi_app:app --host 0.0.0.0 --port 8000 --workers 4` |

**Why workers, and not threads.** Neither development server is single-threaded. Flask's
`--with-threads` is on by default, so every connection gets its own thread; FastAPI hands each
plain `def` handler to a threadpool. Threads are not the limit — the interpreter is. One process
runs one thread's Python at a time, so threads in one process take turns on one core however many
of them there are.

Which is worth measuring rather than believing, because the obvious fix — *use more threads* —
makes it worse. Below is the same total amount of prediction work, split across pools of
different sizes.

In [40]:
import threading

TOTAL = 800
row = pd.DataFrame([[2018.0, 1, 45000.0, 2, 1, 18.5, 1200.0, 85.0, 5.0]], columns=PRICE_FEATURES)

print(f"{'threads':>8}{'preds/s':>10}{'mean ms':>10}{'p95 ms':>9}")

for n_threads in (2, 8, 40):
    per_thread = TOTAL // n_threads
    latencies = []   # list.append is safe from several threads; the GIL is the whole point here

    def one_thread():
        for _ in range(per_thread):
            t0 = time.perf_counter()
            price_model.predict(row)
            latencies.append((time.perf_counter() - t0) * 1000)

    threads = [threading.Thread(target=one_thread) for _ in range(n_threads)]
    t0 = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    wall = time.perf_counter() - t0

    latencies.sort()
    print(f"{n_threads:>8}{TOTAL / wall:>10.0f}"
          f"{sum(latencies) / len(latencies):>10.2f}{latencies[int(0.95 * len(latencies))]:>9.2f}")

 threads   preds/s   mean ms   p95 ms


       2      3238      0.62     0.73


       8      2498      3.05     6.63


      40      1569     14.50    34.34
time: 1.08 s (started: 2026-09-23 20:09:10 +05:30)


Throughput **peaks almost immediately and then falls**, on a machine with plenty of idle cores,
and the latency column is where it really shows: the same prediction that answered in well under
a millisecond takes tens of milliseconds once forty callers are interleaved. Nothing got faster.
The work was the same; it just queued inside one interpreter.

So the threadpool limit is not a ceiling holding you back. It is **backpressure** — a queue is a
better failure mode than forty half-finished predictions. For work like this you want concurrency
*low*, and the thing you add is processes: four **workers** are four interpreters with four
independent locks, so four requests are genuinely served at once.

And here is the part that surprises people: each worker imports the module, and the module body
is where `joblib.load()` lives. **Four workers means four copies of the models in RAM.**

In [41]:
model_mb = sum(p.stat().st_size for p in ARTIFACTS.glob("*.joblib")) / 1024**2

print(f"these artifacts on disk : {model_mb:5.2f} MB")
print(f"cpu cores here          : {os.cpu_count()}")
print()
print("  workers   served at once   these models    a 2 GB model")
for w in (1, 4, 8, 16):
    print(f"  {w:>7}   {w:>13}   {w * model_mb:9.2f} MB   {w * 2.0:8.1f} GB")

these artifacts on disk :  2.41 MB
cpu cores here          : 16

  workers   served at once   these models    a 2 GB model
        1               1        2.41 MB        2.0 GB
        4               4        9.66 MB        8.0 GB
        8               8       19.31 MB       16.0 GB
       16              16       38.63 MB       32.0 GB
time: 736 µs (started: 2026-09-23 20:09:11 +05:30)


For *these* models the multiplication is harmless — sixteen workers is a rounding error in RAM,
so the worker count here is purely a concurrency decision. The column beside it is why the rule
is worth knowing anyway: run the same arithmetic on a model that is large rather than small and
`--workers 16` stops being a throughput setting and becomes an out-of-memory kill.

On-disk size is the right order of magnitude for the RAM cost, not the exact figure — an
unpickled tree ensemble is bigger in memory than on disk. Measure the real thing with `ps` once
the service is up, and check it against the container's limit before raising the count.

A reasonable starting point is `(2 × cores) + 1` for I/O-bound work, and **roughly one worker per
core** for CPU-bound inference like this. Then measure.

```bash
# Flask, production — gunicorn is a real WSGI server; `flask run` is not
pip install gunicorn
gunicorn -w 4 -b 0.0.0.0:5000 flask_app:app

# FastAPI, production
uvicorn fastapi_app:app --host 0.0.0.0 --port 8000 --workers 4
```

`--host 0.0.0.0` is what makes the service reachable from outside the machine; the default
`127.0.0.1` accepts local connections only. That is the right default for a laptop and the wrong
one everywhere else — and inside a container, `127.0.0.1` means *inside that container*, so a
published port would reach nothing.

The ports are back to the conventional `5000` and `8000` here, and that is not an inconsistency:
§3.0 moved off them because *this* machine already has something on both. A container starts with
nothing listening, so the default is free, and the port a deployment publishes is a decision made
outside the application anyway.

## 8.1 The health check that only proves the port is open

Four workers behind a load balancer need one more thing: a way for the load balancer to ask each
one whether to send it traffic. Almost every service in production answers that question like
this:

```python
@app.get("/health")
def health():
    return {"status": "ok"}       # what does this actually prove?
```

It proves that a Python process is running and that its routing table works. Nobody was asking
that. The orchestrator wants to know whether **this process can serve a real request**, and a
process can be listening on the port while:

- `artifacts/` was never copied into the image, so the load failed — except it didn't, because the `FileNotFoundError` guard is at import and a restart loop hides it
- the `.joblib` file is truncated from a half-finished upload
- the pickle loaded under a scikit-learn that renamed an attribute, so `predict` raises on first use
- `model_meta.json` is from a different training run than the model beside it

Through all four, `{"status": "ok"}` keeps saying yes, and every real request fails. Both
services here answer the question that was asked instead: score a fixed canary, report the
threshold in force and both scikit-learn versions, and answer **503** rather than 200 if any of
it fails.

In [42]:
print("healthy:")
print(json.dumps(api.get("/health").json(), indent=2))

healthy:
{
  "status": "ok",
  "models": [
    "price",
    "loan"
  ],
  "canary_price_lakhs": 5.31,
  "decision_threshold": 0.8,
  "model_version": "milepost-2026-09-23",
  "sklearn_version": "1.7.2",
  "sklearn_version_at_train": "1.7.2"
}
time: 4.17 ms (started: 2026-09-23 20:09:11 +05:30)


`sklearn_version` against `sklearn_version_at_train` is the one to watch — it is the difference
between "the model loaded" and "the model loaded under the library that wrote it".

Now break it. This has to be done from inside the process, because the whole point is that
nothing on the outside can tell — which makes it a test technique rather than anything you would
do to a running deployment.

In [43]:
saved_model = flask_app.price_model
flask_app.price_model = None          # the artifact "failed to load", after startup

r = client.get("/health")
print(f"GET /health -> {r.status_code}")
print(json.dumps(r.get_json(), indent=2))
print()
print("a bare {'status': 'ok'} would have returned 200 here, and the load balancer")
print("would have kept sending traffic to a process that cannot answer.")

flask_app.price_model = saved_model
print(f"\nrestored    -> {client.get('/health').status_code}")

GET /health -> 503
{
  "reason": "AttributeError: 'NoneType' object has no attribute 'predict'",
  "status": "unhealthy"
}

a bare {'status': 'ok'} would have returned 200 here, and the load balancer
would have kept sending traffic to a process that cannot answer.

restored    -> 200
time: 3.33 ms (started: 2026-09-23 20:09:11 +05:30)


`503 Service Unavailable` is the correct code and it is the one an orchestrator acts on: stop
routing here, and if it persists, replace the process.

Two vocabulary notes, because the distinction shows up in every deployment tool:

| | Question | Right answer on failure |
|---|---|---|
| **Liveness** | Is this process wedged and in need of a kill? | restart the container |
| **Readiness** | Can this process serve traffic *right now*? | take it out of the pool, leave it running |

The endpoint above is a **readiness** check — it does real work, so a slow or broken model shows
up in it. Polling it once a second means a canary prediction once a second per worker, which for
this model is nothing and for a large one is worth caching or splitting into two endpoints.

Something to restart the process when it dies, and something to ask it whether it is ready: a
container runtime does both, and that is the next step past this notebook.

## 8.2 Readiness, liveness, and the deploy that should never start

`/health` above answers a question *after* startup. There is a second question, asked before it,
and it has a different right answer.

- **Liveness** — is this process alive? Failing it gets the container killed and restarted.
- **Readiness** — can this process serve? Failing it gets the container taken out of the load balancer, but left running.

A truncated `.joblib` is a readiness failure: restarting will not fix it, so a restart loop is
the worst possible response. A hung process is a liveness failure. Wiring both to the same
endpoint is how a bad artifact turns into a crash-loop that pages somebody at 3am.

And then there is a third case, which is not a health check at all: **config that is missing or
wrong.** §5.6 ended on it — both services read the threshold at import, with no fallback. Here
is what that looks like from outside the process, using the `MILEPOST_ARTIFACTS` env var to point
a fresh interpreter at a deliberately incomplete deployment.

In [44]:
import subprocess
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    broken = Path(tmp) / "artifacts"
    broken.mkdir()
    for name in ("price_model.joblib", "loan_model.joblib"):
        shutil.copy(ARTIFACTS / name, broken / name)

    # Everything is here except the one number nobody sends in the payload.
    meta = json.loads((ARTIFACTS / "model_meta.json").read_text())
    del meta["decision_threshold"]
    (broken / "model_meta.json").write_text(json.dumps(meta))

    r = subprocess.run(
        [sys.executable, "-c", "import fastapi_app"],
        cwd=str(Path.cwd()),
        env={**os.environ, "MILEPOST_ARTIFACTS": str(broken)},
        capture_output=True,
        text=True,
    )

print("exit code :", r.returncode)
print("stderr    :", r.stderr.strip().splitlines()[-1])

exit code : 1
stderr    : KeyError: 'decision_threshold'
time: 1.44 s (started: 2026-09-23 20:09:11 +05:30)


The import died. Nothing bound a port, no handler ran, and `gunicorn` or `uvicorn` in front of
this would have exited too — so the orchestrator never marks the new version ready and **the
previous deployment keeps serving**. A configuration mistake became a failed rollout instead of
a service quietly approving people at a cut nobody chose.

Compare that with the version almost everybody writes:

```python
THRESHOLD = META.get("decision_threshold", 0.5)   # looks defensive, is not
```

That line starts cleanly, passes its health check, serves `200`s, and applies a lending policy
that no human agreed to. `.get()` with a default is the right reflex for something optional. A
decision boundary is not optional, and neither is the feature order or the model version beside
it — for anything load-bearing, the crash *is* the feature.

```
P0 setup  P1 models  P2 the web  P3 flask  P4 flask's gaps  P5 fastapi  P6 the seam  P7 async  P8 workers  [ P9 PICK ONE ]
```

# Part 9 — Pick one, then ship it

Two files now serve the same two models. Measured rather than asserted:

In [45]:
rows = []
for path in ("flask_app.py", "fastapi_app.py"):
    src = Path(path).read_text().splitlines()
    body = [ln for ln in src if ln.strip() and not ln.strip().startswith("#")]
    rows.append({"file": path, "lines": len(src), "lines of code": len(body)})

pd.DataFrame(rows)

,file,lines,lines of code
0,flask_app.py,282,204
1,fastapi_app.py,279,211


time: 3.36 ms (started: 2026-09-23 20:09:12 +05:30)


The two files land within a few lines of each other, and that is the result worth noticing: **the
contract is free.** Four Pydantic classes cost about thirty lines, and they pay for themselves
against what the Flask file spends instead — the `if/else` chain that silently defaults, the
`float()` calls that raise, and §4.5's twenty-five-line validator that covered one endpoint out
of two and would have to be written again for the other.

Read the printed numbers rather than remembering a direction; add a route to either file and the
ordering moves. The point is the *slope*, and it is flat.

Most of both files is the model layer they have in common, which is the next thing to talk about.

| | **Flask** | **FastAPI** |
|---|---|---|
| A route is | `@app.route("/x", methods=["POST"])` | `@app.post("/x")` |
| The request arrives as | a plain `dict` | an instance of your class |
| Input validation | **none** | **automatic, from the type annotations** |
| A missing field becomes | `KeyError` → **`500`** | **`422`**, naming the field |
| An unlisted category becomes | a silent default, or `KeyError` → `500` | **`422`**, listing what is allowed |
| `"45000"` becomes | the string `"45000"` | `45000.0` |
| A list of 100 cars, one of them bad | prices the good ones, then `500` — no telling the caller where it stopped | **`422` naming the index**, before any work |
| Docs for the caller | whatever you write and then forget to update | **generated, live at `/docs`** |
| Concurrency model | WSGI, one request per worker thread | ASGI, `async`-capable |
| Production server | `gunicorn -w 4 flask_app:app` | `uvicorn --workers 4` |
| Right when | the service is tiny and you own both ends | **anything another team depends on** |

Flask is not a bad framework — it is a *small* one, and its smallness is the feature. For a
five-line internal endpoint where the caller is you, it is less to think about. The moment
someone else's code depends on the shape of your JSON, the contract stops being optional.

## 9.1 The duplication, and what it becomes

Both service files carry their own copy of the model layer: the same `encode_dict`, the same
nine features, the same `model_pred`. That is deliberate here — one file you can read from top
to bottom is worth more than a file plus a private module you have to go and find.

It is also exactly the duplication that drifts. Change `encode_dict` in one file and not the
other and both services keep answering, with different prices for the same car. Nothing raises.

In a system with a real lifespan that layer becomes one of two things:

- **A package both services import.** `pip install milepost-model`, versioned, with the pins from §1.2 attached to it. The encoding can then only drift by release.
- **A service both of them call.** One process owns the model and the contract; every UI and every integration is a client of it, including the ones that do not exist yet.

The second is what "put the API underneath and make everything else a client" means:

```mermaid
flowchart LR
    B["Browser"] --> UI["Streamlit / Gradio<br/>widgets only"]
    BANK["Partner bank<br/>loan desk"] --> API
    UI -- "POST /predict/price" --> API["FastAPI<br/>validation + contract"]
    API --> M["model_pred / loan_pred"]
    M --> ART[("artifacts/<br/>*.joblib")]
```

One place loads the model. One place owns the contract. A UI becomes a client like any other —
which is what makes the second UI, the mobile app, or the bank's integration cost nothing new.

## 9.2 The decision, in one rule

> **Is the consumer a person or a program?**
>
> A person → a UI framework, and the model loads inside it.
> A program → **FastAPI**. Reach for **Flask** only when the service is tiny and you own both ends.
>
> Once more than one consumer exists, stop choosing: put **FastAPI** underneath and make every UI a client.

Everything else in this notebook is detail in support of that. Both models were finished and on
disk at the end of Part 1, and there were seven Parts to go — which is the same observation from
a different angle: the model was never the product.

In [46]:
print("what is on disk now\n")
for p in sorted(Path(".").glob("*.py")) + sorted(Path(".").glob("*.txt")):
    if not p.name.startswith("_"):
        print(f"  {p.name:<22} {p.stat().st_size / 1024:7.1f} KB")
for p in sorted(DATA.glob("*")) + sorted(ARTIFACTS.glob("*")):
    print(f"  {str(p):<22} {p.stat().st_size / 1024:7.1f} KB")

what is on disk now

  fastapi_app.py            10.1 KB
  flask_app.py               9.5 KB
  requirements.txt           0.2 KB
  data/cars24-car-price.csv  1746.7 KB
  data/train_flask.csv      36.5 KB
  artifacts/loan_model.joblib     1.3 KB
  artifacts/model_meta.json     0.4 KB
  artifacts/price_model.joblib  2470.8 KB
time: 1.1 ms (started: 2026-09-23 20:09:12 +05:30)


```
flask_app.py      flask     ·  flask --app flask_app.py run --port 5001      ·  gunicorn -w 4 flask_app:app
fastapi_app.py    fastapi   ·  uvicorn fastapi_app:app --reload --port 8001  ·  /docs  /redoc  /openapi.json
requirements.txt  pinned with ==, because a pickle expects the version that wrote it
```

Both take the same JSON. Only one of them tells the caller what that JSON has to look like.